# Store Sales - Time Series Forecasting 🛒

This notebook builds a baseline model using LightGBM for the Kaggle "Store Sales - Time Series Forecasting" competition.  
We will:
1. Explore the datasets
2. Create time-based features
3. Train a LightGBM regressor with time-series validation
4. Generate a submission


In [ ]:
import os
from glob import glob
from pathlib import Path

print("Files in input directory:")
for p in sorted(glob("/kaggle/input/store-sales-time-series-forecasting/*")):
    print("-", Path(p).name)


In [ ]:
import pandas as pd

train = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/train.csv", parse_dates=['date'])
test = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/test.csv", parse_dates=['date'])
stores = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/stores.csv")
oil = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/oil.csv", parse_dates=['date'])
holidays = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv", parse_dates=['date'])

print("Train shape:", train.shape)
print("Test shape:", test.shape)
display(train.head())


## Exploratory Data Analysis

Let's quickly check ranges, missing values, and unique IDs in the datasets.


In [ ]:
print("Train date range:", train['date'].min(), "→", train['date'].max())
print("Test date range:", test['date'].min(), "→", test['date'].max())
print("\nStores:", train['store_nbr'].nunique())
print("Families:", train['family'].nunique())

print("\nMissing values in train:")
print(train.isnull().sum())


## Feature Engineering

We will add basic time-based features from the date column such as day, month, year, and day of week.  
These are simple but very effective for seasonal forecasting.


In [ ]:
def create_date_features(df):
    df['day'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['day_of_week'] = df['date'].dt.dayofweek
    df['year'] = df['date'].dt.year
    return df

train = create_date_features(train)
test = create_date_features(test)


## Categorical Encoding

The `family` column is categorical. We will encode it as category type so LightGBM can handle it efficiently.


In [ ]:
train['family'] = train['family'].astype('category')
test['family'] = test['family'].astype('category')


## Validation Strategy

To mimic the forecasting setting, we will train on the history up to the last month, and validate on the last 28 days.  
This prevents leakage and gives a realistic estimate of model performance.


In [ ]:
from sklearn.metrics import mean_squared_log_error
import lightgbm as lgb
import numpy as np

# Use last 4 weeks of train as validation
cutoff_date = train['date'].max() - pd.Timedelta(days=28)
train_set = train[train['date'] <= cutoff_date]
valid_set = train[train['date'] > cutoff_date]

features = ['store_nbr','family','day','month','day_of_week','year']
target = 'sales'

lgb_train = lgb.Dataset(train_set[features], label=train_set[target])
lgb_valid = lgb.Dataset(valid_set[features], label=valid_set[target])

params = {
    "objective":"regression",
    "metric":"rmse",
    "learning_rate":0.1,
    "num_leaves":31,
    "seed":42
}

model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_valid],
    num_boost_round=200,
    callbacks=[
        lgb.early_stopping(stopping_rounds=20),
        lgb.log_evaluation(50)
    ]
)

y_pred_valid = model.predict(valid_set[features], num_iteration=model.best_iteration)
rmsle = mean_squared_log_error(valid_set[target], np.maximum(0,y_pred_valid))**0.5
print("Validation RMSLE:", rmsle)


## Final Training and Predictions

We now retrain the model on the full training data and generate predictions for the test set.


In [ ]:
final_model = lgb.train(
    params,
    lgb.Dataset(train[features], label=train[target]),
    num_boost_round=model.best_iteration
)

test_preds = final_model.predict(test[features])


## Submission

We prepare the submission file with two columns:  
- `id` (from test.csv)  
- `sales` (our predicted values, clipped at zero to avoid negatives)


In [ ]:
submission = pd.DataFrame({
    "id": test['id'],
    "sales": np.maximum(0, test_preds)  # enforce non-negative
})
submission.to_csv("submission.csv", index=False)
print("Submission saved:", submission.shape)
display(submission.head())
